# RBC 통계 시각화

`../stats_*/` 의 최신 `rbc_stats_*.dat` 와 `../log/rbc_gpu_*.log` 를 읽어 그린다.
실행하면 같은 폴더에 PNG 로도 저장된다. (MPM-STD_C `apps/rbc/fig9_data` 의 노트북 관례)

| 그림 | 내용 | 보는 법 |
|---|---|---|
| 1 | $\bar T(z)$ | 벽 경계층 + 중심 등온핵 ≈ 0 (OB 대칭) |
| 2 | $w_{rms}(z)$, $\overline{wT}(z)$ | 플룸 강도 분포 |
| 3 | 총 열유속 $Nu(z)$ | **z 와 무관하게 상수**면 에너지 수지가 닫힌 것 |
| 4 | 시간 이력 $Nu(t)$, $\theta_c(t)$ | Kerr(1996) 8.3~8.4 띠와 비교, 대칭 확인 |

In [1]:
import glob, math, re
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def load_latest_statistics(run_directory):
    """런 폴더에서 가장 마지막 stats 파일을 읽는다 → (헤더 dict, 열 dict)."""
    paths = sorted(glob.glob(f"{run_directory}/rbc_stats_*.dat"))
    if not paths:
        return None
    header, rows = {}, []
    for line in open(paths[-1]):
        if line.startswith("#"):
            for key, value in re.findall(r"(\w+)=([-\d.eE+]+)", line):
                header[key] = float(value)
            continue
        rows.append([float(x) for x in line.split()])
    data = np.array(rows)
    return dict(header=header, path=paths[-1], z=data[:, 0],
                temperature_mean=data[:, 1], w_rms=data[:, 2], wT_mean=data[:, 3])

# 그릴 런 목록 — 새 런을 추가하면 여기 한 줄
runs = {}
for label, directory in [("128x128x64", "../stats_rbc_ra1e6"),
                         ("192x192x96", "../stats_rbc_ra1e6_fine")]:
    loaded = load_latest_statistics(directory)
    if loaded:
        runs[label] = loaded
        h = loaded["header"]
        print(f"{label}: {loaded['path']}  Nu_volume={h['Nu_volume']:.4f} "
              f"Nu_wall_bottom={h['Nu_wall_bottom']:.4f} Nu_wall_top={h['Nu_wall_top']:.4f} "
              f"theta_c={h['theta_c']:+.5f}")
assert runs, "stats 파일이 하나도 없다 — 런을 먼저 돌릴 것"

128x128x64: ../stats_rbc_ra1e6/rbc_stats_00035808.dat  Nu_volume=9.1151 Nu_wall_bottom=9.1111 Nu_wall_top=9.1222 theta_c=-0.00063
192x192x96: ../stats_rbc_ra1e6_fine/rbc_stats_00051844.dat  Nu_volume=8.8740 Nu_wall_bottom=8.8900 Nu_wall_top=8.8736 theta_c=-0.00086


In [2]:
figure, axis = plt.subplots(figsize=(5.2, 4))
for label, run in runs.items():
    axis.plot(run["temperature_mean"], run["z"], "-o", ms=2.5, lw=1, label=label)
axis.axvline(0, color="gray", lw=0.5)
axis.set_xlabel("mean temperature"); axis.set_ylabel("z")
axis.set_title("mean temperature profile (walls +0.5 / -0.5)")
axis.legend(); figure.tight_layout()
figure.savefig("temperature_profile.png", dpi=150); plt.show()

In [3]:
figure, (axis_w, axis_wt) = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for label, run in runs.items():
    axis_w.plot(run["w_rms"], run["z"], lw=1.2, label=label)
    axis_wt.plot(run["wT_mean"], run["z"], lw=1.2, label=label)
axis_w.set_xlabel("w_rms"); axis_w.set_ylabel("z"); axis_w.legend()
axis_wt.set_xlabel("<wT>")
figure.suptitle("vertical velocity rms / convective heat flux")
figure.tight_layout()
figure.savefig("wrms_wT_profile.png", dpi=150); plt.show()

In [4]:
# 총 열유속의 z 불변성 — Nu(z) = (<wT> - alpha dT/dz) / (alpha dT_wall / L)
figure, axis = plt.subplots(figsize=(5.2, 4))
for label, run in runs.items():
    h = run["header"]
    alpha = 1.0 / math.sqrt(h["Ra"] * h["Pr"])
    conductive = -alpha * np.gradient(run["temperature_mean"], run["z"])
    nusselt_of_z = (run["wT_mean"] + conductive) / alpha   # dT=1, L=1
    axis.plot(nusselt_of_z, run["z"], lw=1.2,
              label=f"{label}  (Nu_volume {h['Nu_volume']:.2f})")
axis.set_xlabel("Nu(z) = total heat flux / conductive"); axis.set_ylabel("z")
axis.set_title("flux constancy check — flat line = closed energy budget")
axis.legend(); figure.tight_layout()
figure.savefig("flux_constancy.png", dpi=150); plt.show()

In [5]:
def load_monitor_blocks(log_path):
    """모니터 로그에서 '# step' 헤더로 시작하는 마지막 블록(본런)의 수치 행만."""
    blocks, current = [], None
    for line in open(log_path):
        if line.lstrip().startswith("# step"):
            current = []; blocks.append(current); continue
        parts = line.split()
        if current is not None and len(parts) == 7:
            try: current.append([float(x) for x in parts])
            except ValueError: pass
    return np.array(blocks[-1]) if blocks and blocks[-1] else None

figure, (axis_nu, axis_theta) = plt.subplots(2, 1, figsize=(6.4, 5.6), sharex=True)
for log_path, label in [("../log/rbc_gpu_35325.log", "128x128x64"),
                        ("../log/rbc_gpu_fine_35326.log", "192x192x96")]:
    block = load_monitor_blocks(log_path)
    if block is None: print("skip (no log):", log_path); continue
    time, theta_c, nusselt = block[:, 1], block[:, 4], block[:, 5]
    started = nusselt > 0        # 통계 시작 전 running Nu 는 0
    axis_nu.plot(time[started], nusselt[started], lw=1.2, label=label)
    axis_theta.plot(time, theta_c, lw=0.8, label=label)
axis_nu.axhspan(8.3, 8.4, color="tab:green", alpha=0.25, label="Kerr (1996) 8.3-8.4")
axis_nu.set_ylabel("running Nu"); axis_nu.legend()
axis_theta.axhline(0, color="gray", lw=0.5)
axis_theta.set_xlabel("time (free-fall units)"); axis_theta.set_ylabel("theta_c")
figure.suptitle("Nu / center temperature history  (Ra=1e6, Pr=0.7, Gamma=6)")
figure.tight_layout()
figure.savefig("nusselt_history.png", dpi=150); plt.show()

## fig9 — NOB 글리세롤 θ_c·ΔT vs ΔT (Zhang / Horn / PaScaL_TCS 비교)

NOB 런(`apps/rbc_nob/stats_nob_dt*/`)의 시간평균 T_mean(z) 를 z=0.5 로 보간한
θ_c 에 ΔT[K] 를 곱해 참조 데이터와 겹친다. 참조 재현 목표: **ΔT=40 에서
θ_c=0.1383 (MPM-STD_C P7 = Horn +0.05%)**. OB 런은 대칭이라 θ_c≡0 — 이 그림은
NOB 전용이다.

In [6]:
reference_curves = {}
for name, filename, color, marker in [("Zhang", "zhang.csv", "tab:green", "o"),
                                      ("Horn", "Horm.csv", "tab:orange", "D"),
                                      ("PaScaL_TCS", "PaScaL_TCS.csv", "tab:blue", "s")]:
    reference_curves[name] = dict(data=np.genfromtxt(filename, delimiter=","),
                                  color=color, marker=marker)

def latest_nob_statistics(run_directory):
    paths = sorted(glob.glob(f"{run_directory}/rbc_nob_stats_*.dat"))
    if not paths:
        return None
    header, rows = {}, []
    for line in open(paths[-1]):
        if line.startswith("#"):
            for key, value in re.findall(r"(\w+)=([-\d.eE+]+)", line):
                header[key] = float(value)
            continue
        rows.append([float(x) for x in line.split()])
    data = np.array(rows)
    theta_c = float(np.interp(0.5, data[:, 0], data[:, 1]))  # T_mean @ z=0.5
    return dict(path=paths[-1], delta_temperature=header["DeltaT"], theta_c=theta_c,
                samples=int(header["samples"]))

present = [s for s in (latest_nob_statistics(d)
                       for d in sorted(glob.glob("../../rbc_nob/stats_nob_dt*")))
           if s]
figure, axis = plt.subplots(figsize=(6.2, 4.8))
for name, ref in reference_curves.items():
    if name == "Zhang":
        axis.plot(ref["data"][:, 0], ref["data"][:, 1], color=ref["color"],
                  marker=ref["marker"], ms=3, lw=1, label=name)
    else:
        axis.scatter(ref["data"][:, 0], ref["data"][:, 1], c=ref["color"],
                     marker=ref["marker"], edgecolors="black", linewidths=0.8,
                     zorder=4, label=name)
if present:
    axis.scatter([s["delta_temperature"] for s in present],
                 [s["theta_c"] * s["delta_temperature"] for s in present],
                 s=120, marker="^", c="tab:red", edgecolors="black", alpha=0.75,
                 zorder=5, label="MPM-STD (present)")
    for s in present:
        horn = reference_curves["Horn"]["data"]
        reference_theta = np.interp(s["delta_temperature"], horn[:, 0], horn[:, 1]) \
                          / s["delta_temperature"]
        error = (s["theta_c"] - reference_theta) / reference_theta * 100
        print(f"dT={s['delta_temperature']:g}K: theta_c={s['theta_c']:.4f} "
              f"(x dT = {s['theta_c']*s['delta_temperature']:.3f}K)  vs Horn "
              f"{reference_theta:.4f} ({error:+.2f}%)   [{s['samples']} samples, "
              f"{s['path']}]")
else:
    print("NOB 본런 통계가 아직 없다 — apps/rbc_nob 의 dt40 런 완료 후 재실행")
axis.set_xlabel("$\\Delta T$ [K]"); axis.set_ylabel("$\\theta_c \\cdot \\Delta T$ [K]")
axis.set_title("NOB glycerol center-temperature shift (fig9)")
axis.legend(); figure.tight_layout()
figure.savefig("fig9_theta_c.png", dpi=150); plt.show()

NOB 본런 통계가 아직 없다 — apps/rbc_nob 의 dt40 런 완료 후 재실행
